In [1]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.14.1+cpu
0.29.1+cpu


In [2]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT

model = efficientnet_b0(weights=weights)

model.eval()

print("EfficientNet-B0 loaded successfully")

EfficientNet-B0 loaded successfully


In [3]:
import glob
import os

dataset_path = r"C:\Users\Vinay\Downloads\archive\CUB_200_2011"

image_files = glob.glob(
    os.path.join(dataset_path, "images", "*", "*.jpg")
)

print("Number of images:", len(image_files))
print("First image:", image_files[0])

Number of images: 11796
First image: C:\Users\Vinay\Downloads\archive\CUB_200_2011\images\001.Black_footed_Albatross\Black_Footed_Albatross_0001_796111.jpg


In [4]:
from PIL import Image

img = Image.open(image_files[0]).convert("RGB")

print("Image size:", img.size)

Image size: (320, 223)


In [5]:
transform = weights.transforms()

input_image = transform(img)

print("Tensor shape:", input_image.shape)

Tensor shape: torch.Size([3, 224, 224])


In [6]:
input_batch = input_image.unsqueeze(0)

with torch.no_grad():
    output = model(input_batch)

print("Output shape:", output.shape)

Output shape: torch.Size([1, 1000])


In [7]:
import torch.nn as nn

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    200
)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=200, bias=True)
)


In [1]:
import pandas as pd
import os

dataset_path = r"C:\Users\Vinay\Downloads\archive\CUB_200_2011"

images_df = pd.read_csv(
    os.path.join(dataset_path, "images.txt"),
    sep=" ",
    header=None,
    names=["image_id", "image_path"]
)

bbox_df = pd.read_csv(
    os.path.join(dataset_path, "bounding_boxes.txt"),
    sep=" ",
    header=None,
    names=["image_id", "x", "y", "width", "height"]
)

labels_df = pd.read_csv(
    os.path.join(dataset_path, "image_class_labels.txt"),
    sep=" ",
    header=None,
    names=["image_id", "class_id"]
)

print("Images:", len(images_df))
print("Bounding boxes:", len(bbox_df))
print("Labels:", len(labels_df))

ModuleNotFoundError: No module named 'pandas'

In [2]:
import sys
!{sys.executable} -m pip install pandas

Could not fetch URL https://pypi.org/simple/pandas/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/pandas/ (Caused by SSLError(SSLCertVerificationError('A certificate chain processed, but terminated in a root certificate which is not trusted by the trust provider.'))) - skipping


ERROR: Could not find a version that satisfies the requirement pandas (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for pandas


In [3]:
import sys

!{sys.executable} -m pip install pandas --trusted-host pypi.org --trusted-host files.pythonhosted.org

ERROR: Could not find a version that satisfies the requirement pandas (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for pandas


In [4]:
import os

dataset_path = r"C:\Users\Vinay\Downloads\archive\CUB_200_2011"

# Read images.txt
images = {}

with open(os.path.join(dataset_path, "images.txt"), "r") as f:
    for line in f:
        image_id, image_path = line.strip().split()
        images[int(image_id)] = image_path

# Read bounding_boxes.txt
boxes = {}

with open(os.path.join(dataset_path, "bounding_boxes.txt"), "r") as f:
    for line in f:
        parts = line.strip().split()

        image_id = int(parts[0])
        x = float(parts[1])
        y = float(parts[2])
        width = float(parts[3])
        height = float(parts[4])

        boxes[image_id] = (x, y, width, height)

# Read image_class_labels.txt
labels = {}

with open(os.path.join(dataset_path, "image_class_labels.txt"), "r") as f:
    for line in f:
        image_id, class_id = line.strip().split()
        labels[int(image_id)] = int(class_id)

print("Images:", len(images))
print("Bounding boxes:", len(boxes))
print("Labels:", len(labels))
print("Classes:", len(set(labels.values())))

Images: 11788
Bounding boxes: 11788
Labels: 11788
Classes: 200


In [5]:
import random

random.seed(42)

train_data = []
test_data = []

# Process each of the 200 classes
for class_id in range(1, 201):

    # Get all image IDs belonging to this class
    class_image_ids = [
        image_id
        for image_id, label in labels.items()
        if label == class_id
    ]

    # Shuffle reproducibly
    random.shuffle(class_image_ids)

    # 20 for training
    train_ids = class_image_ids[:20]

    # Next 2 for testing
    test_ids = class_image_ids[20:22]

    for image_id in train_ids:
        train_data.append((image_id, class_id))

    for image_id in test_ids:
        test_data.append((image_id, class_id))

print("Training images:", len(train_data))
print("Testing images:", len(test_data))
print("Total:", len(train_data) + len(test_data))

Training images: 4000
Testing images: 400
Total: 4400


In [6]:
from torch.utils.data import Dataset
from PIL import Image

class CUBDataset(Dataset):

    def __init__(self, data, images, boxes, transform=None):
        self.data = data
        self.images = images
        self.boxes = boxes
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        image_id, class_id = self.data[index]

        # Get image path
        image_path = os.path.join(
            dataset_path,
            "images",
            self.images[image_id]
        )

        # Open image
        image = Image.open(image_path).convert("RGB")

        # Get bounding box
        x, y, width, height = self.boxes[image_id]

        # Convert to integer coordinates
        left = int(x)
        top = int(y)
        right = int(x + width)
        bottom = int(y + height)

        # Crop the bird
        image = image.crop((left, top, right, bottom))

        # Apply preprocessing / augmentation
        if self.transform:
            image = self.transform(image)

        # Convert CUB class IDs 1-200 → labels 0-199
        label = class_id - 1

        return image, label

In [7]:
weights = EfficientNet_B0_Weights.DEFAULT

transform = weights.transforms()

train_dataset = CUBDataset(
    train_data,
    images,
    boxes,
    transform=transform
)

test_dataset = CUBDataset(
    test_data,
    images,
    boxes,
    transform=transform
)

print("Training dataset:", len(train_dataset))
print("Testing dataset:", len(test_dataset))

NameError: name 'EfficientNet_B0_Weights' is not defined

In [8]:
import torch
import torchvision
import os

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image

In [9]:
weights = EfficientNet_B0_Weights.DEFAULT
transform = weights.transforms()

train_dataset = CUBDataset(
    train_data,
    images,
    boxes,
    transform=transform
)

test_dataset = CUBDataset(
    test_data,
    images,
    boxes,
    transform=transform
)

print("Training dataset:", len(train_dataset))
print("Testing dataset:", len(test_dataset))

Training dataset: 4000
Testing dataset: 400


In [10]:
image, label = train_dataset[0]

print("Image tensor:", image.shape)
print("Label:", label)

Image tensor: torch.Size([3, 224, 224])
Label: 0


In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 125
Testing batches: 13


In [12]:
images_batch, labels_batch = next(iter(train_loader))

print("Images:", images_batch.shape)
print("Labels:", labels_batch.shape)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])


In [13]:
for param in model.features.parameters():
    param.requires_grad = False

print("EfficientNet feature layers frozen.")

NameError: name 'model' is not defined

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT

model = efficientnet_b0(weights=weights)

# Change ImageNet's 1000 classes to CUB's 200 classes
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    200
)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=200, bias=True)
)


In [15]:
for param in model.features.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

print("CNN frozen. Classifier trainable.")

CNN frozen. Classifier trainable.


In [16]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print("Device:", device)

Device: cpu


In [1]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images_batch, labels_batch in train_loader:

        images_batch = images_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = model(images_batch)

        loss = criterion(outputs, labels_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels_batch.size(0)
        correct += (predicted == labels_batch).sum().item()

    train_accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {running_loss / len(train_loader):.4f} "
        f"Train Accuracy: {train_accuracy:.2f}%"
    )

NameError: name 'model' is not defined

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Load pretrained EfficientNet-B0
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Change ImageNet's 1000 classes to CUB's 200 classes
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    200
)

# Freeze CNN feature extractor
for param in model.features.parameters():
    param.requires_grad = False

# Keep classifier trainable
for param in model.classifier.parameters():
    param.requires_grad = True

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer — only train classifier
optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

print("Model ready!")
print("Device:", device)
print("Output classes:", model.classifier[1].out_features)

Model ready!
Device: cpu
Output classes: 200


In [3]:
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

NameError: name 'train_loader' is not defined

In [4]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

class CUBDataset(Dataset):

    def __init__(self, data, images, boxes, transform=None):
        self.data = data
        self.images = images
        self.boxes = boxes
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        image_id, class_id = self.data[index]

        image_path = os.path.join(
            dataset_path,
            "images",
            self.images[image_id]
        )

        image = Image.open(image_path).convert("RGB")

        x, y, width, height = self.boxes[image_id]

        left = int(x)
        top = int(y)
        right = int(x + width)
        bottom = int(y + height)

        image = image.crop((left, top, right, bottom))

        if self.transform:
            image = self.transform(image)

        label = class_id - 1

        return image, label


# EfficientNet preprocessing
transform = weights.transforms()

# Create datasets
train_dataset = CUBDataset(
    train_data,
    images,
    boxes,
    transform=transform
)

test_dataset = CUBDataset(
    test_data,
    images,
    boxes,
    transform=transform
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

NameError: name 'train_data' is not defined

In [5]:
# ============================================================
# METHOD 2 - COMPLETE SETUP
# ============================================================

import os
import random
import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


# ------------------------------------------------------------
# 1. Dataset path
# ------------------------------------------------------------

dataset_path = r"C:\Users\Vinay\Downloads\archive\CUB_200_2011"


# ------------------------------------------------------------
# 2. Read CUB metadata
# ------------------------------------------------------------

images = {}

with open(os.path.join(dataset_path, "images.txt"), "r") as f:
    for line in f:
        image_id, image_path = line.strip().split()
        images[int(image_id)] = image_path


boxes = {}

with open(os.path.join(dataset_path, "bounding_boxes.txt"), "r") as f:
    for line in f:
        parts = line.strip().split()

        image_id = int(parts[0])
        x = float(parts[1])
        y = float(parts[2])
        width = float(parts[3])
        height = float(parts[4])

        boxes[image_id] = (x, y, width, height)


labels = {}

with open(os.path.join(dataset_path, "image_class_labels.txt"), "r") as f:
    for line in f:
        image_id, class_id = line.strip().split()
        labels[int(image_id)] = int(class_id)


print("Images:", len(images))
print("Bounding boxes:", len(boxes))
print("Labels:", len(labels))
print("Classes:", len(set(labels.values())))


# ------------------------------------------------------------
# 3. Create exact 20 train + 2 test split
# ------------------------------------------------------------

random.seed(42)

train_data = []
test_data = []

for class_id in range(1, 201):

    class_image_ids = [
        image_id
        for image_id, label in labels.items()
        if label == class_id
    ]

    random.shuffle(class_image_ids)

    train_ids = class_image_ids[:20]
    test_ids = class_image_ids[20:22]

    for image_id in train_ids:
        train_data.append((image_id, class_id))

    for image_id in test_ids:
        test_data.append((image_id, class_id))


print("Training images:", len(train_data))
print("Testing images:", len(test_data))


# ------------------------------------------------------------
# 4. EfficientNet weights and preprocessing
# ------------------------------------------------------------

weights = EfficientNet_B0_Weights.DEFAULT
transform = weights.transforms()


# ------------------------------------------------------------
# 5. Dataset class
# ------------------------------------------------------------

class CUBDataset(Dataset):

    def __init__(self, data, images, boxes, transform=None):
        self.data = data
        self.images = images
        self.boxes = boxes
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        image_id, class_id = self.data[index]

        image_path = os.path.join(
            dataset_path,
            "images",
            self.images[image_id]
        )

        image = Image.open(image_path).convert("RGB")

        x, y, width, height = self.boxes[image_id]

        left = int(x)
        top = int(y)
        right = int(x + width)
        bottom = int(y + height)

        image = image.crop(
            (left, top, right, bottom)
        )

        if self.transform:
            image = self.transform(image)

        # Convert 1-200 → 0-199
        label = class_id - 1

        return image, label


# ------------------------------------------------------------
# 6. Create datasets
# ------------------------------------------------------------

train_dataset = CUBDataset(
    train_data,
    images,
    boxes,
    transform=transform
)

test_dataset = CUBDataset(
    test_data,
    images,
    boxes,
    transform=transform
)


# ------------------------------------------------------------
# 7. Create DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)


# ------------------------------------------------------------
# 8. Load EfficientNet-B0
# ------------------------------------------------------------

model = efficientnet_b0(weights=weights)


# ------------------------------------------------------------
# 9. Replace 1000 ImageNet classes with 200 CUB classes
# ------------------------------------------------------------

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    200
)


# ------------------------------------------------------------
# 10. Freeze CNN feature extractor
# ------------------------------------------------------------

for param in model.features.parameters():
    param.requires_grad = False


# Keep final classifier trainable
for param in model.classifier.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 11. Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)


# ------------------------------------------------------------
# 12. Loss + optimizer
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)


# ------------------------------------------------------------
# 13. Final verification
# ------------------------------------------------------------

print("\n========== SETUP COMPLETE ==========")
print("Training images :", len(train_dataset))
print("Testing images  :", len(test_dataset))
print("Training batches:", len(train_loader))
print("Testing batches :", len(test_loader))
print("Device          :", device)
print("Output classes  :", model.classifier[1].out_features)

Images: 11788
Bounding boxes: 11788
Labels: 11788
Classes: 200
Training images: 4000
Testing images: 400

========== SETUP COMPLETE ==========
Training images : 4000
Testing images  : 400
Training batches: 125
Testing batches : 13
Device          : cpu
Output classes  : 200


In [6]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images_batch, labels_batch in train_loader:

        images_batch = images_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = model(images_batch)

        loss = criterion(outputs, labels_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels_batch.size(0)
        correct += (predicted == labels_batch).sum().item()

    train_accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {running_loss / len(train_loader):.4f} "
        f"Train Accuracy: {train_accuracy:.2f}%"
    )

Epoch [1/3] Loss: 4.6627 Train Accuracy: 13.82%
Epoch [2/3] Loss: 2.8976 Train Accuracy: 59.00%
Epoch [3/3] Loss: 2.0269 Train Accuracy: 73.92%


In [7]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images_batch, labels_batch in test_loader:

        images_batch = images_batch.to(device)
        labels_batch = labels_batch.to(device)

        outputs = model(images_batch)

        _, predicted = torch.max(outputs, 1)

        total += labels_batch.size(0)
        correct += (predicted == labels_batch).sum().item()

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 50.25%


In [8]:
# First freeze everything
for param in model.features.parameters():
    param.requires_grad = False

# Unfreeze the last 2 feature blocks
for param in model.features[-2:].parameters():
    param.requires_grad = True

# Keep classifier trainable
for param in model.classifier.parameters():
    param.requires_grad = True

print("Last 2 EfficientNet blocks are now trainable.")

Last 2 EfficientNet blocks are now trainable.


In [9]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

print("Fine-tuning optimizer ready.")

Fine-tuning optimizer ready.


In [10]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images_batch, labels_batch in train_loader:

        images_batch = images_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = model(images_batch)

        loss = criterion(outputs, labels_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels_batch.size(0)
        correct += (predicted == labels_batch).sum().item()

    train_accuracy = 100 * correct / total

    print(
        f"Fine-tune Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {running_loss / len(train_loader):.4f} "
        f"Train Accuracy: {train_accuracy:.2f}%"
    )

Fine-tune Epoch [1/3] Loss: 1.3794 Train Accuracy: 82.55%
Fine-tune Epoch [2/3] Loss: 1.0824 Train Accuracy: 85.55%
Fine-tune Epoch [3/3] Loss: 0.9094 Train Accuracy: 87.97%


In [11]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images_batch, labels_batch in test_loader:

        images_batch = images_batch.to(device)
        labels_batch = labels_batch.to(device)

        outputs = model(images_batch)

        _, predicted = torch.max(outputs, 1)

        total += labels_batch.size(0)
        correct += (predicted == labels_batch).sum().item()

test_accuracy = 100 * correct / total

print(f"Fine-tuned Test Accuracy: {test_accuracy:.2f}%")

Fine-tuned Test Accuracy: 55.00%


In [12]:
torch.save(
    model.state_dict(),
    "efficientnet_b0_cub_finetuned.pth"
)

print("Model saved successfully!")

Model saved successfully!


In [13]:
results = {
    "Method 1 - RGB Histogram + HOG + Linear SVM": 13.50,
    "Method 2 - EfficientNet Stage 1": 50.25,
    "Method 2 - EfficientNet Fine-tuned": 55.00
}

print(results)

{'Method 1 - RGB Histogram + HOG + Linear SVM': 13.5, 'Method 2 - EfficientNet Stage 1': 50.25, 'Method 2 - EfficientNet Fine-tuned': 55.0}


In [16]:
import torch
from PIL import Image

model.eval()

image_id, true_class = test_data[0]

image_path = os.path.join(
    dataset_path,
    "images",
    images[image_id]
)

image = Image.open(image_path).convert("RGB")

# Crop using CUB bounding box
x, y, width, height = boxes[image_id]

image = image.crop((
    int(x),
    int(y),
    int(x + width),
    int(y + height)
))

input_image = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(input_image)
    probabilities = torch.softmax(output, dim=1)

    predicted_class = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0, predicted_class].item()

print("True class ID     :", true_class)
print("Predicted class ID:", predicted_class + 1)
print("Confidence        :", round(confidence * 100, 2), "%")

True class ID     : 1
Predicted class ID: 1
Confidence        : 39.58 %


In [17]:
# Load CUB class names

class_names = {}

with open(os.path.join(dataset_path, "classes.txt"), "r") as f:
    for line in f:
        class_id, class_name = line.strip().split(" ", 1)
        class_names[int(class_id)] = class_name

predicted_name = class_names[predicted_class + 1]
true_name = class_names[true_class]

print("True Bird      :", true_name)
print("Predicted Bird :", predicted_name)
print("Confidence     :", round(confidence * 100, 2), "%")

True Bird      : 001.Black_footed_Albatross
Predicted Bird : 001.Black_footed_Albatross
Confidence     : 39.58 %


In [18]:
correct = 0
total = 10

model.eval()

for i in range(total):
    image_id, true_class = test_data[i]

    image_path = os.path.join(
        dataset_path,
        "images",
        images[image_id]
    )

    image = Image.open(image_path).convert("RGB")

    x, y, width, height = boxes[image_id]

    image = image.crop((
        int(x),
        int(y),
        int(x + width),
        int(y + height)
    ))

    input_image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_image)
        probabilities = torch.softmax(output, dim=1)

        predicted_class = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0, predicted_class].item()

    predicted_id = predicted_class + 1

    if predicted_id == true_class:
        correct += 1

    print(
        f"{i+1}. True: {class_names[true_class]} | "
        f"Predicted: {class_names[predicted_id]} | "
        f"Confidence: {confidence*100:.2f}%"
    )

print("\nCorrect:", correct, "/", total)
print("Accuracy on these 10 images:", correct / total * 100, "%")

1. True: 001.Black_footed_Albatross | Predicted: 001.Black_footed_Albatross | Confidence: 39.58%
2. True: 001.Black_footed_Albatross | Predicted: 087.Mallard | Confidence: 6.64%
3. True: 002.Laysan_Albatross | Predicted: 084.Red_legged_Kittiwake | Confidence: 29.08%
4. True: 002.Laysan_Albatross | Predicted: 002.Laysan_Albatross | Confidence: 63.82%
5. True: 003.Sooty_Albatross | Predicted: 001.Black_footed_Albatross | Confidence: 23.95%
6. True: 003.Sooty_Albatross | Predicted: 003.Sooty_Albatross | Confidence: 46.01%
7. True: 004.Groove_billed_Ani | Predicted: 004.Groove_billed_Ani | Confidence: 57.09%
8. True: 004.Groove_billed_Ani | Predicted: 033.Yellow_billed_Cuckoo | Confidence: 4.94%
9. True: 005.Crested_Auklet | Predicted: 005.Crested_Auklet | Confidence: 18.38%
10. True: 005.Crested_Auklet | Predicted: 137.Cliff_Swallow | Confidence: 34.22%

Correct: 5 / 10
Accuracy on these 10 images: 50.0 %
